In [ ]:
from langchain_ollama import ChatOllama
from langchain_ollama import OllamaEmbeddings
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from typing import Annotated, TypedDict, List
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, BaseMessage
from langgraph.prebuilt import ToolNode, tools_condition
import time
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
import re

In [ ]:
# Data + Index

loader = PyPDFLoader("intro-to-ml.pdf", mode="page")
docs = loader.load()

In [ ]:
len(docs)

In [ ]:
llm = ChatOllama(
    model="granite4.1:3b",
    base_url="http://127.0.0.1:11434",
    num_predict=512,
    keep_alive="1m",
    client_kwargs={"timeout": 120},
    seed=42,
    top_k=20,
    top_p=0.8,
    repeat_penalty=1.1
)

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

chunks = splitter.split_documents(docs)

for d in chunks:
    d.page_content = d.page_content.encode("utf-8", "ignore").decode("utf-8", "ignore")

In [ ]:
embedding_1 = OllamaEmbeddings(
    model="nomic-embed-text:v1.5",
    base_url="http://127.0.0.1:11434",
    keep_alive=60,
    client_kwargs={"timeout": 120},
)

valid_chunks = [d for d in chunks if d.page_content.strip()]

if not valid_chunks:
    raise ValueError("No text found in chunks.")

vector_store = None
batch_size = 8

for start in range(0, len(valid_chunks), batch_size):
    batch = valid_chunks[start:start + batch_size]

    print(f"Processing chunks {start + 1}–{start + len(batch)}...", flush=True)

    if vector_store is None:
        vector_store = FAISS.from_documents(batch, embedding_1)
    else:
        vector_store.add_documents(batch)

print("Index ready!")

retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 5})

In [ ]:
# State

class State(TypedDict):
    question: str
    docs: List[Document]
    good_docs: List[Document]
    verdict: str
    reason: str

    strips: List[str]
    kept_strips: List[str]
    refined_context: str

    web_query: str
    
    web_docs: List[Document]

    answer: str

In [ ]:
# Retrieve

def retrieve_node(state: State) -> State:

    q = state["question"]
    
    return {"docs": retriever.invoke(q)}

In [ ]:
# Score-based doc evaluator

class DocEvalScore(BaseModel):

    score: float = Field(ge=0.0, le=1.0, description="Relevance score between 0 and 1")

    reason: str


doc_eval_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a strict retrieval evaluator for RAG.\n"
            "You will be given ONE retrieved chunk and a question.\n"
            "Return a relevance score in [0.0, 1.0].\n"
            "- 1.0: chunk alone is sufficient to answer fully/mostly\n"
            "- 0.0: chunk is irrelevant\n"
            "Be conservative with high scores.\n"
            "For comparison questions, give a high score only if the chunk explains both concepts and their differences.\n"
            "A chunk discussing only one concept provides partial support and should not receive a high score.\n"
            "Also return a short reason.\n"
            "Output JSON only."
            "Ignore page numbers, headers, footers, and chapter titles when scoring.\n"
            "A topic name alone is not evidence. If the body does not help answer the question, score 0.0.\n"
            "For a definition question, a clear definition is sufficient; algorithms or comparisons are not required.\n"
        ),
        ("human", "Question: {question}\n\nChunk:\n{chunk}"),
    ]
)

doc_eval_chain = doc_eval_prompt | llm.with_structured_output(DocEvalScore)


UPPER_TH = 0.7
LOWER_TH = 0.3


def eval_each_doc_node(state: State) -> State:

    q = state["question"]

    if not state["docs"]:
        return {
            "good_docs": [],
            "verdict": "INCORRECT",
            "reason": "No documents were retrieved.",
        }
    
    scores: List[float] = []
    good: List[Document] = []

    for d in state["docs"]:
        # Remove this PDF's chapter footer before scoring
        clean_text = re.sub(r"\b\d+\s*\|\s*Chapter\s+\d+\s*:[^\n]*", "", d.page_content).strip()

        out = doc_eval_chain.invoke({"question": q, "chunk": clean_text})
        
        scores.append(out.score)

        print(
            f"Page: {d.metadata.get('page', '?')} | "
            f"Score: {out.score:.2f} | Reason: {out.reason}"
        )

        # Keep any doc above LOWER_TH as "weakly relevant"
        if out.score > LOWER_TH:
            good.append(d)

    # CORRECT: at least one doc > UPPER_TH
    if any(s > UPPER_TH for s in scores):
        return {
            "good_docs": good,
            "verdict": "CORRECT",
            "reason": f"At least one retrieved chunk scored > {UPPER_TH}.",
        }

    # INCORRECT: all docs < LOWER_TH
    if len(scores) > 0 and all(s < LOWER_TH for s in scores):
        return {
            "good_docs": [],
            "verdict": "INCORRECT",
            "reason": f"All retrieved chunks scored < {LOWER_TH}.",
        }

    # AMBIGUOUS: otherwise
    return {
        "good_docs": good,
        "verdict": "AMBIGUOUS",
        "reason": f"No chunk scored > {UPPER_TH}, but not all were < {LOWER_TH}.",
    }

In [ ]:
# -----------------------------
# Sentence-level DECOMPOSER: Split Documents in sentences to put related ones and remove unrelated ones.
# -----------------------------
def decompose_to_sentences(text: str) -> List[str]:
    text = re.sub(r"\s+", " ", text).strip()
    sentences = re.split(r"(?<=[.!?])\s+", text)
    return [s.strip() for s in sentences if s.strip()]

# -----------------------------
# FILTER (LLM judge)
# -----------------------------
class KeepOrDrop(BaseModel):
    keep: bool


filter_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a strict relevance filter.\n"
            "Return keep=true only if the sentence directly helps answer the question.\n"
            "Use ONLY the sentence. Output JSON only.",
        ),
        ("human", "Question: {question}\n\nSentence:\n{sentence}"),
    ]
)

filter_chain = filter_prompt | llm.with_structured_output(KeepOrDrop)


# -----------------------------
# Knowledge refinement
# (CORRECT => internal only)
# (INCORRECT => web only)
# (AMBIGUOUS => internal + web)
# -----------------------------
def refine(state: State) -> State:

    q = state["question"]

    if state.get("verdict") == "CORRECT":
        docs_to_use = state["good_docs"]

    elif state.get("verdict") == "INCORRECT":
        docs_to_use = state["web_docs"]
        
    else:  # AMBIGUOUS
        docs_to_use = state["good_docs"] + state["web_docs"]

    strips = []
    for doc in docs_to_use:
        strips.extend(decompose_to_sentences(doc.page_content))

    kept: List[str] = []
    for s in strips:
        if filter_chain.invoke({"question": q, "sentence": s}).keep:
            kept.append(s)

    refined_context = "\n".join(kept).strip()

    return {
        "strips": strips,
        "kept_strips": kept,
        "refined_context": refined_context,
    }

    # Here we are splitting Documents into sentences and checking whether each sentence is related to Query or not.
    # If related than put it otherwise left it and continue.

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults

# Query rewrite for web search

class WebQuery(BaseModel):
    query: str

rewrite_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Rewrite the user question into a web search query composed of keywords.\n"
            "Rules:\n"
            "- Keep it short (6–14 words).\n"
            "- If the question implies recency (e.g., recent/latest/last week/last month), add a constraint like (last 30 days).\n"
            "- Do NOT answer the question.\n"
            "- Return JSON with a single key: query",
        ),
        ("human", "Question: {question}"),
    ]
)

rewrite_chain = rewrite_prompt | llm.with_structured_output(WebQuery)


def rewrite_query_node(state: State) -> State:

    out = rewrite_chain.invoke({"question": state["question"]})

    return {"web_query": out.query}


# --------------------------------
# Web search node: uses web_query
# --------------------------------
tavily = TavilySearchResults(max_results=5)


def web_search_node(state: State) -> State:
    q = state.get("web_query") or state["question"]

    try:
        results = tavily.invoke({"query": q})

        # Your current TavilySearchResults should return a list
        if not isinstance(results, list):
            raise ValueError("Unexpected web-search response.")

        web_docs = []

        for r in results:
            if not isinstance(r, dict):
                continue

            content = r.get("content") or r.get("snippet") or ""
            if not content.strip():
                continue

            title = r.get("title", "")
            url = r.get("url", "")

            web_docs.append(Document(
                page_content=content,
                metadata={"url": url, "title": title},
            ))

        return {"web_docs": web_docs}

    except Exception as exc:
        print(f"Web search failed: {type(exc).__name__}: {exc}")
        return {"web_docs": []}

In [ ]:
# Generate

answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful ML tutor. Answer ONLY using the provided context.\n"
            "If the context is empty or insufficient, say: 'I don't know.'",
        ),
        ("human", "Question: {question}\n\nContext:\n{context}"),
    ]
)


def generate(state: State) -> State:

    context = state["refined_context"].strip()

    if not context:
        return {"answer": "I couldn't find enough information to answer reliably."}

    out = (answer_prompt | llm).invoke({"question": state["question"], "context": context})

    return {"answer": out.content}

In [ ]:
# -----------------------------
# Routing
# CORRECT => refine
# INCORRECT / AMBIGUOUS => rewrite -> web_search -> refine -> generate
# -----------------------------
def route_after_eval(state: State) -> str:
    if state["verdict"] == "CORRECT":
        return "refine"
    else:
        return "rewrite_query"

In [ ]:
# -----------------------------
# Build graph
# -----------------------------
g = StateGraph(State)

g.add_node("retrieve", retrieve_node)
g.add_node("eval_each_doc", eval_each_doc_node)

g.add_node("rewrite_query", rewrite_query_node)
g.add_node("web_search", web_search_node)

g.add_node("refine", refine)
g.add_node("generate", generate)

g.add_edge(START, "retrieve")
g.add_edge("retrieve", "eval_each_doc")

g.add_conditional_edges(
    "eval_each_doc",
    route_after_eval,
    {
        "refine": "refine",
        "rewrite_query": "rewrite_query",
    },
)

# non-correct path
g.add_edge("rewrite_query", "web_search")
g.add_edge("web_search", "refine")

# correct path already goes to refine
g.add_edge("refine", "generate")
g.add_edge("generate", END)

app = g.compile()

app

In [ ]:
# -----------------------------
# Run example
# -----------------------------
res = app.invoke(
    {
        "question": "Batch normalization vs layer normalization",
        "docs": [],
        "good_docs": [],
        "verdict": "",
        "reason": "",
        "strips": [],
        "kept_strips": [],
        "refined_context": "",
        "web_query": "",
        "web_docs": [],
        "answer": "",
    }
)

print("VERDICT:", res["verdict"])
print("REASON:", res["reason"])
print("WEB_QUERY:", res["web_query"])
print("\nREFINED CONTEXT:\n", res["refined_context"])
print("\nOUTPUT:\n", res["answer"])

In [ ]:
# -----------------------------
# Run example
# -----------------------------
res = app.invoke(
    {
        "question": "What is supervised learning?",
        "docs": [],
        "good_docs": [],
        "verdict": "",
        "reason": "",
        "strips": [],
        "kept_strips": [],
        "refined_context": "",
        "web_query": "",
        "web_docs": [],
        "answer": "",
    }
)

print("VERDICT:", res["verdict"])
print("REASON:", res["reason"])
print("WEB_QUERY:", res["web_query"])
print("\nREFINED CONTEXT:\n", res["refined_context"])
print("\nOUTPUT:\n", res["answer"])

In [ ]:
# -----------------------------
# Run example
# -----------------------------
res = app.invoke(
    {
        "question": "What is the difference between classification and regression?",
        "docs": [],
        "good_docs": [],
        "verdict": "",
        "reason": "",
        "strips": [],
        "kept_strips": [],
        "refined_context": "",
        "web_query": "",
        "web_docs": [],
        "answer": "",
    }
)

print("VERDICT:", res["verdict"])
print("REASON:", res["reason"])
print("WEB_QUERY:", res["web_query"])
print("\nREFINED CONTEXT:\n", res["refined_context"])
print("\nOUTPUT:\n", res["answer"])

In [ ]:
# -----------------------------
# Run example
# -----------------------------
res = app.invoke(
    {
        "question": "How does a Transformer use self-attention?",
        "docs": [],
        "good_docs": [],
        "verdict": "",
        "reason": "",
        "strips": [],
        "kept_strips": [],
        "refined_context": "",
        "web_query": "",
        "web_docs": [],
        "answer": "",
    }
)

print("VERDICT:", res["verdict"])
print("REASON:", res["reason"])
print("WEB_QUERY:", res["web_query"])
print("\nREFINED CONTEXT:\n", res["refined_context"])
print("\nOUTPUT:\n", res["answer"])

In [ ]:
# -----------------------------
# Run example
# -----------------------------
res = app.invoke(
    {
        "question": "What is the difference between batch normalization and layer normalization?",
        "docs": [],
        "good_docs": [],
        "verdict": "",
        "reason": "",
        "strips": [],
        "kept_strips": [],
        "refined_context": "",
        "web_query": "",
        "web_docs": [],
        "answer": "",
    }
)

print("VERDICT:", res["verdict"])
print("REASON:", res["reason"])
print("WEB_QUERY:", res["web_query"])
print("\nREFINED CONTEXT:\n", res["refined_context"])
print("\nOUTPUT:\n", res["answer"])